In [0]:
acs_raw = spark.read.csv(
    "/Volumes/workspace/default/dic_project/ACSDP5Y2024.DP03-2026-04-16T012621.csv",
    header=True,
    inferSchema=True
)

print(f"Rows: {acs_raw.count()}")
print(f"Columns: {len(acs_raw.columns)}")
display(acs_raw)

Rows: 145
Columns: 209


Label (Grouping),Alabama!!Estimate,Alabama!!Margin of Error,Alabama!!Percent,Alabama!!Percent Margin of Error,Alaska!!Estimate,Alaska!!Margin of Error,Alaska!!Percent,Alaska!!Percent Margin of Error,Arizona!!Estimate,Arizona!!Margin of Error,Arizona!!Percent,Arizona!!Percent Margin of Error,Arkansas!!Estimate,Arkansas!!Margin of Error,Arkansas!!Percent,Arkansas!!Percent Margin of Error,California!!Estimate,California!!Margin of Error,California!!Percent,California!!Percent Margin of Error,Colorado!!Estimate,Colorado!!Margin of Error,Colorado!!Percent,Colorado!!Percent Margin of Error,Connecticut!!Estimate,Connecticut!!Margin of Error,Connecticut!!Percent,Connecticut!!Percent Margin of Error,Delaware!!Estimate,Delaware!!Margin of Error,Delaware!!Percent,Delaware!!Percent Margin of Error,District of Columbia!!Estimate,District of Columbia!!Margin of Error,District of Columbia!!Percent,District of Columbia!!Percent Margin of Error,Florida!!Estimate,Florida!!Margin of Error,Florida!!Percent,Florida!!Percent Margin of Error,Georgia!!Estimate,Georgia!!Margin of Error,Georgia!!Percent,Georgia!!Percent Margin of Error,Hawaii!!Estimate,Hawaii!!Margin of Error,Hawaii!!Percent,Hawaii!!Percent Margin of Error,Idaho!!Estimate,Idaho!!Margin of Error,Idaho!!Percent,Idaho!!Percent Margin of Error,Illinois!!Estimate,Illinois!!Margin of Error,Illinois!!Percent,Illinois!!Percent Margin of Error,Indiana!!Estimate,Indiana!!Margin of Error,Indiana!!Percent,Indiana!!Percent Margin of Error,Iowa!!Estimate,Iowa!!Margin of Error,Iowa!!Percent,Iowa!!Percent Margin of Error,Kansas!!Estimate,Kansas!!Margin of Error,Kansas!!Percent,Kansas!!Percent Margin of Error,Kentucky!!Estimate,Kentucky!!Margin of Error,Kentucky!!Percent,Kentucky!!Percent Margin of Error,Louisiana!!Estimate,Louisiana!!Margin of Error,Louisiana!!Percent,Louisiana!!Percent Margin of Error,Maine!!Estimate,Maine!!Margin of Error,Maine!!Percent,Maine!!Percent Margin of Error,Maryland!!Estimate,Maryland!!Margin of Error,Maryland!!Percent,Maryland!!Percent Margin of Error,Massachusetts!!Estimate,Massachusetts!!Margin of Error,Massachusetts!!Percent,Massachusetts!!Percent Margin of Error,Michigan!!Estimate,Michigan!!Margin of Error,Michigan!!Percent,Michigan!!Percent Margin of Error,Minnesota!!Estimate,Minnesota!!Margin of Error,Minnesota!!Percent,Minnesota!!Percent Margin of Error,Mississippi!!Estimate,Mississippi!!Margin of Error,Mississippi!!Percent,Mississippi!!Percent Margin of Error,Missouri!!Estimate,Missouri!!Margin of Error,Missouri!!Percent,Missouri!!Percent Margin of Error,Montana!!Estimate,Montana!!Margin of Error,Montana!!Percent,Montana!!Percent Margin of Error,Nebraska!!Estimate,Nebraska!!Margin of Error,Nebraska!!Percent,Nebraska!!Percent Margin of Error,Nevada!!Estimate,Nevada!!Margin of Error,Nevada!!Percent,Nevada!!Percent Margin of Error,New Hampshire!!Estimate,New Hampshire!!Margin of Error,New Hampshire!!Percent,New Hampshire!!Percent Margin of Error,New Jersey!!Estimate,New Jersey!!Margin of Error,New Jersey!!Percent,New Jersey!!Percent Margin of Error,New Mexico!!Estimate,New Mexico!!Margin of Error,New Mexico!!Percent,New Mexico!!Percent Margin of Error,New York!!Estimate,New York!!Margin of Error,New York!!Percent,New York!!Percent Margin of Error,North Carolina!!Estimate,North Carolina!!Margin of Error,North Carolina!!Percent,North Carolina!!Percent Margin of Error,North Dakota!!Estimate,North Dakota!!Margin of Error,North Dakota!!Percent,North Dakota!!Percent Margin of Error,Ohio!!Estimate,Ohio!!Margin of Error,Ohio!!Percent,Ohio!!Percent Margin of Error,Oklahoma!!Estimate,Oklahoma!!Margin of Error,Oklahoma!!Percent,Oklahoma!!Percent Margin of Error,Oregon!!Estimate,Oregon!!Margin of Error,Oregon!!Percent,Oregon!!Percent Margin of Error,Pennsylvania!!Estimate,Pennsylvania!!Margin of Error,Pennsylvania!!Percent,Pennsylvania!!Percent Margin of Error,Rhode Island!!Estimate,Rhode Island!!Margin of Error,Rhode Island!!Percent,Rhode Island!!Percent Margin of Error,South C

In [0]:
from pyspark.sql.functions import col

# show just the label column to find our variables
labels = acs_raw.select("Label (Grouping)").collect()
for i, row in enumerate(labels):
    print(i, row[0])

0 EMPLOYMENT STATUS
1     Population 16 years and over
2         In labor force
3             Civilian labor force
4                 Employed
5                 Unemployed
6             Armed Forces
7         Not in labor force
8     Civilian labor force
9         Unemployment Rate
10     Females 16 years and over
11         In labor force
12             Civilian labor force
13                 Employed
14     Own children of the householder under 6 years
15         All parents in family in labor force
16     Own children of the householder 6 to 17 years
17         All parents in family in labor force
18 COMMUTING TO WORK
19     Workers 16 years and over
20         Car, truck, or van -- drove alone
21         Car, truck, or van -- carpooled
22         Public transportation
23         Walked
24         Other means
25         Worked from home
26         Mean travel time to work (minutes)
27 OCCUPATION
28     Civilian employed population 16 years and over
29         Management, business, sc

In [0]:
import pandas as pd


acs_pandas = acs_raw.toPandas()

state_columns = [col for col in acs_pandas.columns if '!!Estimate' in col]

# extract the rows we need
rows_needed = {
    'Unemployment_Rate': 9,
    'Median_Household_Income': 67,
    'Uninsured_Rate': 105,
    'Poverty_Rate': 135
}

result = {}
for var_name, row_idx in rows_needed.items():
    result[var_name] = acs_pandas.loc[row_idx, state_columns].values

# build clean dataframe
state_names = [col.replace('!!Estimate', '') for col in state_columns]
acs_clean = pd.DataFrame(result, index=state_names).reset_index()
acs_clean.columns = ['State_Name'] + list(rows_needed.keys())

print(acs_clean.head(10))
print(f"Shape: {acs_clean.shape}")

             State_Name Unemployment_Rate  ... Uninsured_Rate Poverty_Rate
0               Alabama               (X)  ...        459,841          (X)
1                Alaska               (X)  ...         77,711          (X)
2               Arizona               (X)  ...        763,416          (X)
3              Arkansas               (X)  ...        269,851          (X)
4            California               (X)  ...      2,557,944          (X)
5              Colorado               (X)  ...        446,581          (X)
6           Connecticut               (X)  ...        192,081          (X)
7              Delaware               (X)  ...         63,563          (X)
8  District of Columbia               (X)  ...         24,454          (X)
9               Florida               (X)  ...      2,548,782          (X)

[10 rows x 5 columns]
Shape: (52, 5)


In [0]:
print(acs_pandas.columns[:9].tolist())

['Label (Grouping)', 'Alabama!!Estimate', 'Alabama!!Margin of Error', 'Alabama!!Percent', 'Alabama!!Percent Margin of Error', 'Alaska!!Estimate', 'Alaska!!Margin of Error', 'Alaska!!Percent', 'Alaska!!Percent Margin of Error']


In [0]:
# use Percent columns for rates, Estimate for median income
percent_columns = [col for col in acs_pandas.columns if '!!Percent' in col and 'Margin' not in col]
estimate_columns = [col for col in acs_pandas.columns if '!!Estimate' in col]

# extract each variable from the correct column type
acs_clean = pd.DataFrame({
    'State_Name': [col.replace('!!Percent', '') for col in percent_columns],
    'Unemployment_Rate': acs_pandas.loc[9, percent_columns].values,
    'Poverty_Rate': acs_pandas.loc[135, percent_columns].values,
    'Uninsured_Rate': acs_pandas.loc[105, percent_columns].values,
    'Median_Household_Income': acs_pandas.loc[67, estimate_columns].values,
})

# clean up remove commas and convert to numeric
for col in ['Unemployment_Rate', 'Poverty_Rate', 'Uninsured_Rate', 'Median_Household_Income']:
    acs_clean[col] = pd.to_numeric(
        acs_clean[col].astype(str).str.replace(',', '').str.replace('%', ''),
        errors='coerce'
    )

print(acs_clean.head(10))
print(f"Shape: {acs_clean.shape}")

             State_Name  ...  Median_Household_Income
0               Alabama  ...                    63999
1                Alaska  ...                    92788
2               Arizona  ...                    79964
3              Arkansas  ...                    60773
4            California  ...                    99122
5              Colorado  ...                    95470
6           Connecticut  ...                    95781
7              Delaware  ...                    84954
8  District of Columbia  ...                   109870
9               Florida  ...                    74568

[10 rows x 5 columns]
Shape: (52, 5)


In [0]:
print(acs_clean[['State_Name', 'Unemployment_Rate', 'Poverty_Rate', 'Uninsured_Rate']].head(10))
print(acs_clean.isnull().sum())

             State_Name  Unemployment_Rate  Poverty_Rate  Uninsured_Rate
0               Alabama                4.7          15.6             9.2
1                Alaska                5.9          10.1            11.0
2               Arizona                5.1          12.5            10.5
3              Arkansas                4.9          16.0             9.0
4            California                6.6          12.0             6.6
5              Colorado                4.6           9.4             7.7
6           Connecticut                5.6          10.0             5.4
7              Delaware                5.1          10.4             6.3
8  District of Columbia                6.3          15.4             3.6
9               Florida                4.8          12.6            11.5
State_Name                 0
Unemployment_Rate          0
Poverty_Rate               0
Uninsured_Rate             0
Median_Household_Income    0
dtype: int64


In [0]:
acs_spark = spark.createDataFrame(acs_clean)

acs_spark.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_acs")

print("ACS bronze table saved successfully")
print(f"Rows: {acs_spark.count()}")
display(acs_spark)

ACS bronze table saved successfully
Rows: 52


State_Name,Unemployment_Rate,Poverty_Rate,Uninsured_Rate,Median_Household_Income
Alabama,4.7,15.6,9.2,63999
Alaska,5.9,10.1,11.0,92788
Arizona,5.1,12.5,10.5,79964
Arkansas,4.9,16.0,9.0,60773
California,6.6,12.0,6.6,99122
Colorado,4.6,9.4,7.7,95470
Connecticut,5.6,10.0,5.4,95781
Delaware,5.1,10.4,6.3,84954
District of Columbia,6.3,15.4,3.6,109870
Florida,4.8,12.6,11.5,74568


In [0]:
gold_df = spark.read.table("workspace.default.gold_overdose")
acs_df = spark.read.table("workspace.default.bronze_acs")

# join on State_Name
joined_df = gold_df.join(acs_df, on="State_Name", how="inner")

print(f"Gold rows: {gold_df.count()}")
print(f"ACS rows: {acs_df.count()}")
print(f"Joined rows: {joined_df.count()}")
display(joined_df)

Gold rows: 45813
ACS rows: 52
Joined rows: 43393


State_Name,State,Indicator,Date,Death_Count,Percent_Complete,Percent_Pending_Investigation,Predicted_Value,year,month,quarter,lag_1,lag_3,roll_mean_3,roll_std_3,Unemployment_Rate,Poverty_Rate,Uninsured_Rate,Median_Household_Income
Alaska,AK,Cocaine (T40.5),2016-07-01,13.0,100.0,0.0,13.0,2016,7,3,11.0,11.0,11.666666666666666,1.1547005383792517,5.9,10.1,11.0,92788
Alaska,AK,Cocaine (T40.5),2016-08-01,12.0,100.0,0.02338634238,12.0,2016,8,3,13.0,11.0,12.0,1.0,5.9,10.1,11.0,92788
Alaska,AK,Cocaine (T40.5),2016-09-01,11.0,100.0,0.0469924812,11.0,2016,9,3,12.0,11.0,12.0,1.0,5.9,10.1,11.0,92788
Alaska,AK,Cocaine (T40.5),2016-10-01,13.0,100.0,0.06991377301,13.0,2016,10,4,11.0,13.0,12.0,1.0,5.9,10.1,11.0,92788
Alaska,AK,Cocaine (T40.5),2016-11-01,15.0,100.0,0.06994637445,15.0,2016,11,4,13.0,12.0,13.0,2.0,5.9,10.1,11.0,92788
Alaska,AK,Cocaine (T40.5),2016-12-01,15.0,100.0,0.06888633754,15.0,2016,12,4,15.0,11.0,14.333333333333334,1.1547005383792517,5.9,10.1,11.0,92788
Alaska,AK,Cocaine (T40.5),2017-01-01,15.0,100.0,0.06858710562,15.0,2017,1,1,15.0,13.0,15.0,0.0,5.9,10.1,11.0,92788
Alaska,AK,Cocaine (T40.5),2017-02-01,18.0,100.0,0.06901311249,18.0,2017,2,1,15.0,15.0,16.0,1.7320508075688772,5.9,10.1,11.0,92788
Alaska,AK,Cocaine (T40.5),2017-03-01,19.0,100.0,0.06882312457,19.0,2017,3,1,18.0,15.0,17.333333333333332,2.0816659994661326,5.9,10.1,11.0,92788
Alaska,AK,Cocaine (T40.5),2017-04-01,16.0,100.0,0.06936416185,16.0,2017,4,2,19.0,15.0,17.666666666666668,1.5275252316519465,5.9,10.1,11.0,92788


In [0]:
# save joined data as gold_combined Delta table
joined_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_combined")

print("gold_combined table saved successfully")
print(f"Total rows: {joined_df.count()}")

gold_combined table saved successfully
Total rows: 43393


In [0]:
from pyspark.sql.functions import avg, corr, round, desc

# Insight 1: Do high poverty states have higher overdose death counts?
print("Insight 1: Poverty Rate vs Average Overdose Deaths")
insight1 = joined_df.filter(
    joined_df.Indicator == "Number of Drug Overdose Deaths"
).groupBy("State_Name", "Poverty_Rate").agg(
    round(avg("Death_Count"), 1).alias("Avg_Death_Count")
).orderBy(desc("Poverty_Rate"))

display(insight1)

Insight 1: Poverty Rate vs Average Overdose Deaths


State_Name,Poverty_Rate,Avg_Death_Count
Puerto Rico,40.5,734.0
Louisiana,18.9,1581.5
Mississippi,18.9,512.7
New Mexico,17.8,725.5
West Virginia,16.7,1049.6
Kentucky,16.1,1719.9
Arkansas,16.0,457.7
Alabama,15.6,1048.6
District of Columbia,15.4,407.5
Oklahoma,15.3,867.9


In [0]:
# Insight 2: correlation between uninsured rate and overdose deaths
# states with less healthcare access may have higher overdose deaths
print("Insight 2: Uninsured Rate vs Overdose Deaths Correlation")

insight2 = joined_df.filter(
    joined_df.Indicator == "Number of Drug Overdose Deaths"
).groupBy("State_Name", "Uninsured_Rate").agg(
    round(avg("Death_Count"), 1).alias("Avg_Death_Count")
).orderBy(desc("Uninsured_Rate"))

# compute correlation
correlation = joined_df.filter(
    joined_df.Indicator == "Number of Drug Overdose Deaths"
).select(corr("Uninsured_Rate", "Death_Count")).collect()[0][0]

print(f"Correlation between Uninsured Rate and Death Count: {correlation:.3f}")
display(insight2)

Insight 2: Uninsured Rate vs Overdose Deaths Correlation
Correlation between Uninsured Rate and Death Count: 0.139


State_Name,Uninsured_Rate,Avg_Death_Count
Texas,17.1,3911.7
Oklahoma,12.9,867.9
Georgia,12.4,1825.9
Florida,11.5,5917.9
Nevada,11.4,923.7
Wyoming,11.3,94.5
Alaska,11.0,201.1
Mississippi,11.0,512.7
Arizona,10.5,2323.6
Tennessee,10.0,2642.5


In [0]:
# Insight 3: deaths per capita: normalizing by state population size
# this addresses our phase 1 finding that raw counts were biased by population size
print("Insight 3: Overdose Deaths Per Capita by State")

from pyspark.sql.functions import col

insight3 = joined_df.filter(
    joined_df.Indicator == "Number of Drug Overdose Deaths"
).groupBy("State_Name", "Median_Household_Income").agg(
    round(avg("Death_Count"), 1).alias("Avg_Death_Count")
).orderBy(desc("Avg_Death_Count"))

# correlation between income and deaths
correlation_income = joined_df.filter(
    joined_df.Indicator == "Number of Drug Overdose Deaths"
).select(corr("Median_Household_Income", "Death_Count")).collect()[0][0]

print(f"Correlation between Median Income and Death Count: {correlation_income:.3f}")
display(insight3)

Insight 3: Overdose Deaths Per Capita by State
Correlation between Median Income and Death Count: 0.037


State_Name,Median_Household_Income,Avg_Death_Count
California,99122,7641.5
Florida,74568,5917.9
Pennsylvania,77971,4573.0
Ohio,71389,4388.3
Texas,78476,3911.7
New York,85974,3088.2
Illinois,83390,2931.4
Tennessee,69595,2642.5
New Jersey,103556,2554.2
Michigan,72875,2542.3


In [0]:
# Combined Model: predict overdose deaths using both overdose history + ACS features
# this is the model that uses features from both datasets

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

combined_df = spark.read.table("workspace.default.gold_combined")

# train/test split
train_combined = combined_df.filter(col("year") < 2023)
test_combined = combined_df.filter(col("year") >= 2023)

print(f"Train: {train_combined.count()}, Test: {test_combined.count()}")

# feature pipeline same as before but now adding ACS features
state_indexer = StringIndexer(inputCol="State", outputCol="State_idx", handleInvalid="keep")
indicator_indexer = StringIndexer(inputCol="Indicator", outputCol="Indicator_idx", handleInvalid="keep")
state_encoder = OneHotEncoder(inputCol="State_idx", outputCol="State_vec")
indicator_encoder = OneHotEncoder(inputCol="Indicator_idx", outputCol="Indicator_vec")

# include ACS features alongside our existing features
assembler = VectorAssembler(
    inputCols=[
        "State_vec", "Indicator_vec",
        "year", "month", "quarter",
        "lag_1", "lag_3", "roll_mean_3", "roll_std_3",
        "Unemployment_Rate", "Poverty_Rate", 
        "Uninsured_Rate", "Median_Household_Income"
    ],
    outputCol="features_raw",
    handleInvalid="keep"
)

scaler = StandardScaler(inputCol="features_raw", outputCol="features")

ridge_combined = LinearRegression(
    featuresCol="features",
    labelCol="Death_Count",
    regParam=1.0,
    elasticNetParam=0.0
)

pipeline_combined = Pipeline(stages=[
    state_indexer, indicator_indexer,
    state_encoder, indicator_encoder,
    assembler, scaler, ridge_combined
])

# train
model_combined = pipeline_combined.fit(train_combined)

# evaluate
evaluator_rmse = RegressionEvaluator(labelCol="Death_Count", predictionCol="prediction", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol="Death_Count", predictionCol="prediction", metricName="mae")
evaluator_r2 = RegressionEvaluator(labelCol="Death_Count", predictionCol="prediction", metricName="r2")

test_preds = model_combined.transform(test_combined)

print("Combined Model (Ridge + ACS features) Results:")
print(f"  Test RMSE: {evaluator_rmse.evaluate(test_preds):.3f}")
print(f"  Test MAE: {evaluator_mae.evaluate(test_preds):.3f}")
print(f"  Test R2: {evaluator_r2.evaluate(test_preds):.3f}")

Train: 29511, Test: 13882
Combined Model (Ridge + ACS features) Results:
  Test RMSE: 77.764
  Test MAE: 28.723
  Test R2: 0.995


In [0]:
# save combined model
model_combined.write().overwrite().save("/Volumes/workspace/default/dic_project/models/combined_model")

print("Combined model saved successfully")

Combined model saved successfully
